# Final 3: economics-driven yield curve forecasting

This notebook tests whether features built from what the literature says moves the yield curve can beat final2's feature set. Same anti-leakage rules, same evaluation dates as final2, so the only difference is the information the model sees.

What the literature says (full detail in `final3_explained.md`):
- daily moves are mostly unforecastable news (CPI/payroll surprises, FOMC) - Gurkaynak-Sack-Swanson 2005
- what is predictable: yield change momentum at ~1 month lags (Sihvonen 2024), post-FOMC drift over ~50 days (Brooks-Katz-Lustig 2018), short end anchoring to EFFR, slope/curvature mean reversion (never level), slow drift toward a trend inflation + policy anchor (Bauer-Rudebusch 2020)

Honest disclosure: this holdout was already inspected while building earlier versions, so every number below is conditioned on prior looks. The 3-month forward test is the only unconditioned evidence.

Verdict from running this notebook: **final3 underperforms final2 at every horizon**, so final2 stays the headline model and this notebook becomes the robustness section showing the economics features are already spanned by final2's set.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

#works whether jupyter starts in the repo root or in notebooks/
if Path("data/FRB_H15.csv").exists():
    dataDir = Path("data")
else:
    dataDir = Path("../data")

#the maturity names match final2's saved forecast files (needed for the head to head join),
#which is why they are 1M/3M/6M instead of the 0Y1M style used in the baseline notebooks
matList = ["1M", "3M", "6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]
maturityYears = np.array([1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30])
dnsFactors = ["level", "slope", "curvature"]

horizons = [1, 5, 20]
maxHorizon = 20

#model selection grids, identical to final2 so any difference is from the features
decayCandidates = np.arange(0.02, 0.801, 0.005)
ridgeCandidates = [0.1, 1.0, 10.0, 100.0, 1000.0]
windowCandidates = [504, 1000, None]
weightCandidates = [0.0, 0.25, 0.50, 0.75, 1.0]
numTuningOrigins = 40
tuningSpacing = 5
trailingWindow = 60
minTrailing = 10

#economics feature settings, each tied to a cited mechanism (see markdown below)
momentumLookbacks = [1, 5, 21, 63]
policyMoveThreshold = 0.075
#exponential decay with a ~50 day e-folding time (half life is about 35 days),
#matching the ~50 trading day post-move drift horizon in brooks-katz-lustig
driftDecayDays = 50
zWindow = 250
istarHalfLife = 756

#### Load data

All merges are by calendar date. final2's saved forecasts are loaded too so this notebook evaluates at exactly the same origins.

In [2]:
#the H15 file has one date column and then the 11 maturities in order short to long
rawYields = pd.read_csv(dataDir / "FRB_H15.csv")
rawYields = rawYields.rename(columns={rawYields.columns[0]: "Date",
    rawYields.columns[1]: "1M", rawYields.columns[2]: "3M", rawYields.columns[3]: "6M",
    rawYields.columns[4]: "1Y", rawYields.columns[5]: "2Y", rawYields.columns[6]: "3Y",
    rawYields.columns[7]: "5Y", rawYields.columns[8]: "7Y", rawYields.columns[9]: "10Y",
    rawYields.columns[10]: "20Y", rawYields.columns[11]: "30Y"})
rawYields["Date"] = pd.to_datetime(rawYields["Date"], errors="coerce")
rawYields = rawYields.set_index("Date")

yieldDf = rawYields[matList]
yieldDf = yieldDf.apply(pd.to_numeric, errors="coerce")
yieldDf = yieldDf.dropna()
yieldDf = yieldDf.sort_index()

breakevenDf = pd.read_csv(dataDir / "T5YIE.csv")
breakevenDf = breakevenDf.rename(columns={"observation_date": "Date", "T5YIE": "breakeven"})
breakevenDf["Date"] = pd.to_datetime(breakevenDf["Date"])
breakevenDf = breakevenDf.set_index("Date")
breakevenDf = breakevenDf[["breakeven"]]
breakevenDf = breakevenDf.apply(pd.to_numeric, errors="coerce")
breakevenDf = breakevenDf.dropna()
breakevenDf = breakevenDf.sort_index()

effrDf = pd.read_csv(dataDir / "EFFR.csv")
effrDf = effrDf.rename(columns={"observation_date": "Date", "EFFR": "effr"})
effrDf["Date"] = pd.to_datetime(effrDf["Date"])
effrDf = effrDf.set_index("Date")
effrDf = effrDf[["effr"]]
effrDf = effrDf.apply(pd.to_numeric, errors="coerce")
effrDf = effrDf.dropna()
effrDf = effrDf.sort_index()

spilloverDf = pd.read_csv(dataDir / "generated" / "spillover_features.csv", parse_dates=["date"])
spilloverDf = spilloverDf.set_index("date")

final2Forecasts = pd.read_csv(dataDir / "generated" / "final2_results" / "yield_forecasts.csv",
    parse_dates=["origin_date", "target_date"])

print("complete curves:", len(yieldDf))
print("range:", yieldDf.index.min().date(), "to", yieldDf.index.max().date())
print("final2 forecast rows:", len(final2Forecasts))

complete curves: 6199
range: 2001-07-31 to 2026-05-14
final2 forecast rows: 113751


#### DNS factors with training-only decay

Same training-only procedure as final2, re-derived here so this notebook runs on its own. For a fixed decay the loadings are a constant 11x3 matrix, and each day's factors are just an OLS fit of that day's 11 yields onto the loadings (pinv is the OLS projection). Day t factors use only day t yields, so this step cannot leak.

In [3]:
def nsLoadings(maturities, decay):
    #one [level, slope, curvature] loading row per maturity
    scaled = decay * maturities
    slopeLoad = (1.0 - np.exp(-scaled)) / scaled
    curveLoad = slopeLoad - np.exp(-scaled)
    levelLoad = np.ones(len(maturities))
    return np.column_stack([levelLoad, slopeLoad, curveLoad])


def estimateDnsFactors(yieldFrame, decay):
    #per-date cross sectional OLS: factors = pinv(L) @ yields
    loadingMatrix = nsLoadings(maturityYears, decay)
    projection = np.linalg.pinv(loadingMatrix)
    factorArray = yieldFrame.to_numpy() @ projection.T
    reconstructed = factorArray @ loadingMatrix.T

    factorsDf = pd.DataFrame(factorArray, index=yieldFrame.index, columns=dnsFactors)
    residualsDf = yieldFrame - pd.DataFrame(reconstructed, index=yieldFrame.index, columns=matList)
    return factorsDf, residualsDf, loadingMatrix


#align on macro availability the same way final2 does, then keep the first 80% for decay selection
aligned = yieldDf.join(breakevenDf, how="inner")
aligned = aligned.join(effrDf, how="inner")
aligned = aligned.join(spilloverDf[["total_connectedness"]], how="inner")
aligned = aligned.dropna()

decayTrainEnd = int(0.80 * len(aligned))
decayTrainDates = aligned.index[:decayTrainEnd]

#guard: decay selection must never touch the evaluated period. this passes on the
#current data vintage and fails loudly if a csv refresh pushes the 80% boundary
#into the frozen holdout
firstEvaluatedOrigin = final2Forecasts["origin_date"].min()
assert decayTrainDates[-1] < firstEvaluatedOrigin, "decay window reaches into evaluated origins"

decayTrainYields = yieldDf.loc[decayTrainDates]

decayRows = []
for candidate in decayCandidates:
    factorsTemp, residualsTemp, loadTemp = estimateDnsFactors(decayTrainYields, candidate)
    fitRmse = 100.0 * np.sqrt(np.mean(residualsTemp.to_numpy() ** 2))
    decayRows.append([candidate, fitRmse])

decayResults = pd.DataFrame(decayRows, columns=["decay", "trainCurveRmseBp"])
bestRow = decayResults["trainCurveRmseBp"].idxmin()
selectedDecay = decayResults["decay"].iloc[bestRow]

dnsDf, residualsDf, loadingMatrix = estimateDnsFactors(yieldDf, selectedDecay)

print("selected decay (training only):", round(selectedDecay, 4))
print("decay training ends:", decayTrainDates[-1].date())

selected decay (training only): 0.485
decay training ends: 2021-09-08


#### Economics features

Every feature dated t only uses data through t (diffs, trailing windows, trailing EWMAs).

| block | mechanism | expected horizon |
|---|---|---|
| momentum (1/5/21/63d factor changes) | yield change autocorrelation, Sihvonen 2024 | 5d, 20d |
| post move drift (~50 day e-folding decay) | underreaction to policy moves, Brooks-Katz-Lustig 2018 | 5d, 20d |
| short end EFFR basis | money market yields pinned to the policy rate | 5d, 20d short end |
| policy cycle state | fed moves come in runs, Coibion-Gorodnichenko 2012 | all (state) |
| shape z-scores | slope/curvature mean revert over weeks, level does not | 20d |
| i-star gap | level reverts to a slow trend inflation + policy anchor, Bauer-Rudebusch 2020 | 20d |
| breakeven momentum | inflation expectation shifts feed nominal yields | 5d, 20d |
| spillover (compact) | cross maturity transmission intensity | all |

Two details: rolling z-score denominators are floored at 1bp (the zero-rate years would otherwise turn 1bp wiggles into huge z-scores), and subtracting series with different date indexes aligns on the union and sprinkles NaN, so the basis is computed with dropna before any rolling window (a rolling window that touches one NaN returns NaN).

Real-time availability convention: a series value dated t is treated as known at the day-t origin. This matches final2 and is exact for the H15 close, but the NY Fed publishes day-t EFFR (and FRED posts T5YIE) with a short lag, so a strict live system would only have effr(t-1) at the day-t close. EFFR is administratively pinned between FOMC moves, so the gap is at most a one-day timing slip around move days - the forward test must use effr and breakeven lagged one day.

In [4]:
econDf = pd.DataFrame(index=dnsDf.index)

#block 1: multi scale momentum, diff(k) at date t is factor(t) - factor(t-k)
for factor in dnsFactors:
    for lookback in momentumLookbacks:
        econDf[factor + "_momentum_" + str(lookback) + "d"] = dnsDf[factor].diff(lookback)

#current factor levels stay in as anchors (deeper lags are redundant once diffs are present)
for factor in dnsFactors:
    econDf[factor + "_current"] = dnsDf[factor]

#block 2 and 4: policy rate moves, post move drift, and cycle state
effrSeries = effrDf["effr"]
effrChange = effrSeries.diff()
isMove = effrChange.abs() >= policyMoveThreshold
moveSign = np.sign(effrChange).where(isMove)

#sign of the most recent move, carried forward (0 before any move)
lastMoveSign = moveSign.ffill().fillna(0.0)

#trading days since the last move: counter minus the counter frozen at the last move
counter = pd.Series(np.arange(len(effrSeries)), index=effrSeries.index)
counterAtMove = counter.where(isMove).ffill().fillna(0)
daysSinceMove = counter - counterAtMove

#post move drift: exponential decay with a ~50 day e-folding time (half life ~35 days)
postMoveDrift = lastMoveSign * np.exp(-daysSinceMove / driftDecayDays)

#signed run of consecutive same-direction moves, +3 means three hikes in a row
runValues = []
currentRun = 0.0
for signValue in moveSign.to_numpy():
    if not np.isnan(signValue):
        if np.sign(currentRun) == signValue:
            currentRun = currentRun + signValue
        else:
            currentRun = signValue
    runValues.append(currentRun)
policyRun = pd.Series(runValues, index=moveSign.index)

policyDf = pd.DataFrame({
    "lastPolicyMoveSign": lastMoveSign,
    "postMoveDrift": postMoveDrift,
    "policyRunSigned": policyRun,
    "effrChange63d": effrSeries.diff(63),
    "effrChange126d": effrSeries.diff(126),
})
econDf = econDf.join(policyDf, how="left")

#block 3: short end EFFR basis
#dropna first: the union alignment leaves NaN where only one series printed, and
#a 250 day rolling window that touches one NaN returns NaN
shortEndMean = yieldDf[["1M", "3M", "6M", "1Y"]].mean(axis=1)
basis = (shortEndMean - effrSeries).dropna()

basisMean = basis.rolling(zWindow).mean()
basisStd = basis.rolling(zWindow).std().clip(lower=0.01)
econDf["shortEndBasis"] = basis
econDf["shortEndBasisZ"] = (basis - basisMean) / basisStd
econDf["shortEndBasisChange5d"] = basis.diff(5)

#block 5: shape mean reversion z-scores (never on level, it is near a unit root)
def trailingZscore(series):
    rollMean = series.rolling(zWindow).mean()
    rollStd = series.rolling(zWindow).std().clip(lower=0.01)
    return (series - rollMean) / rollStd

econDf["slopeZ"] = trailingZscore(dnsDf["slope"])
econDf["curvatureZ"] = trailingZscore(dnsDf["curvature"])
butterfly = 2.0 * yieldDf["5Y"] - yieldDf["2Y"] - yieldDf["10Y"]
econDf["butterflyZ"] = trailingZscore(butterfly)

#block 6: i-star gap, a slow moving nominal anchor (trailing EWMAs are causal)
istar = effrSeries.ewm(halflife=istarHalfLife).mean() + breakevenDf["breakeven"].ewm(halflife=istarHalfLife).mean()
econDf = econDf.join(istar.rename("istar"), how="left")
econDf["istarGap"] = dnsDf["level"] - econDf["istar"]
econDf = econDf.drop(columns=["istar"])

#block 7: breakeven momentum
econDf = econDf.join(breakevenDf["breakeven"].diff(5).rename("breakevenChange5d"), how="left")
econDf = econDf.join(breakevenDf["breakeven"].diff(21).rename("breakevenChange21d"), how="left")

#block 8: compact spillover
longNet = spilloverDf[["net_10Y", "net_20Y", "net_30Y"]].mean(axis=1)
econDf = econDf.join(spilloverDf["total_connectedness"], how="left")
econDf = econDf.join(longNet.rename("longNetTransmission"), how="left")
econDf["connectednessChange5d"] = econDf["total_connectedness"].diff(5)
econDf["longNetChange5d"] = econDf["longNetTransmission"].diff(5)

econDf = econDf.replace([np.inf, -np.inf], np.nan)
featureNames = list(econDf.columns)
print("economics features:", len(featureNames))
print(featureNames)

economics features: 33
['level_momentum_1d', 'level_momentum_5d', 'level_momentum_21d', 'level_momentum_63d', 'slope_momentum_1d', 'slope_momentum_5d', 'slope_momentum_21d', 'slope_momentum_63d', 'curvature_momentum_1d', 'curvature_momentum_5d', 'curvature_momentum_21d', 'curvature_momentum_63d', 'level_current', 'slope_current', 'curvature_current', 'lastPolicyMoveSign', 'postMoveDrift', 'policyRunSigned', 'effrChange63d', 'effrChange126d', 'shortEndBasis', 'shortEndBasisZ', 'shortEndBasisChange5d', 'slopeZ', 'curvatureZ', 'butterflyZ', 'istarGap', 'breakevenChange5d', 'breakevenChange21d', 'total_connectedness', 'longNetTransmission', 'connectednessChange5d', 'longNetChange5d']


#### Modeling table and shared origins

The evaluated origins are exactly final2's origin dates, so the head to head is forecast-for-forecast on identical days. The assertion fails loudly if the features silently shrank the sample.

In [5]:
modelDf = dnsDf.join(econDf, how="inner", rsuffix="_dup")
modelDf = modelDf.join(yieldDf.add_prefix("observed_"), how="inner")
modelDf = modelDf.join(residualsDf.add_prefix("residual_"), how="inner")
modelDf = modelDf.dropna().sort_index()

observedCols = []
for mat in matList:
    observedCols.append("observed_" + mat)

#final2's evaluated origin dates define the comparison sample
final2OriginDates = sorted(final2Forecasts["origin_date"].unique())

datePosition = {}
for position in range(len(modelDf.index)):
    datePosition[modelDf.index[position]] = position

forecastOrigins = []
for originDate in final2OriginDates:
    if originDate in datePosition:
        position = datePosition[originDate]
        if position + maxHorizon < len(modelDf):
            forecastOrigins.append(position)
forecastOrigins = sorted(forecastOrigins)

overlapShare = len(forecastOrigins) / len(final2OriginDates)
print("final2 origins:", len(final2OriginDates), " shared origins:", len(forecastOrigins),
      " overlap:", round(100 * overlapShare, 1), "%")
assert overlapShare > 0.95, "final3 features removed too many origins, comparison would be unfair"

holdoutStart = forecastOrigins[0]
print("first shared origin:", modelDf.index[forecastOrigins[0]].date())
print("last 20 day target:", modelDf.index[forecastOrigins[-1] + maxHorizon].date())
print("aligned rows:", len(modelDf))

final2 origins: 1149  shared origins: 1149  overlap: 100.0 %
first shared origin: 2021-09-09
last 20 day target: 2026-05-14
aligned rows: 5820


#### Supervised construction (change space)

Targets are factor changes f(t+h) - f(t). Under ridge shrinkage a change-space model collapses to "no change" (the random walk) instead of the training-average level, which is the correct default for persistent yields. A training sample for origin t and horizon h may only contain origins o with o + h <= t, so every training target is realized by the time the forecast is made.

In [6]:
predictorDf = econDf.loc[modelDf.index].copy()

#one change target table per horizon: row t holds factors(t+h) - factors(t)
changeTargets = {}
for horizon in horizons:
    changeTargets[horizon] = modelDf[dnsFactors].shift(-horizon) - modelDf[dnsFactors]


def makeTrainingSlice(horizon, origin, windowLength):
    #last usable training origin is origin - horizon (its target lands exactly on the origin)
    lastTrain = origin - horizon
    firstTrain = 0
    if windowLength is not None:
        firstTrain = max(firstTrain, lastTrain - windowLength + 1)
    xTrain = predictorDf.iloc[firstTrain : lastTrain + 1]
    yTrain = changeTargets[horizon].iloc[firstTrain : lastTrain + 1].to_numpy(dtype=float)
    return xTrain, yTrain


def forecastResiduals(origin, horizon):
    #AR(1) per maturity on the DNS fit residuals, re-estimated from the trailing 1000 days
    forecasts = []
    historyStart = max(0, origin - 1000 + 1)
    for mat in matList:
        history = modelDf["residual_" + mat].iloc[historyStart : origin + 1].to_numpy()
        lagged = history[:-1]
        current = history[1:]
        design = np.column_stack([np.ones(len(lagged)), lagged])
        coeffs = np.linalg.lstsq(design, current, rcond=None)[0]
        intercept = coeffs[0]
        persistence = np.clip(coeffs[1], -0.99, 0.99)
        if abs(1.0 - persistence) > 1e-8:
            longRunMean = intercept / (1.0 - persistence)
        else:
            longRunMean = history.mean()
        forecasts.append(longRunMean + persistence ** horizon * (history[-1] - longRunMean))
    return np.array(forecasts)


def fitRidgeAndPredict(horizon, origin, windowLength, ridgeAlpha):
    #scaler and ridge written out in full so each step is visible
    xTrain, yTrain = makeTrainingSlice(horizon, origin, windowLength)
    scaler = StandardScaler()
    xTrainScaled = scaler.fit_transform(xTrain)
    ridgeModel = Ridge(alpha=ridgeAlpha)
    ridgeModel.fit(xTrainScaled, yTrain)
    xCurrent = predictorDf.iloc[[origin]]
    xCurrentScaled = scaler.transform(xCurrent)
    predictedChange = ridgeModel.predict(xCurrentScaled)[0]
    return predictedChange


#sanity check: the change target must equal a hand computed difference
checkOrigin = forecastOrigins[0]
handChange = (modelDf[dnsFactors].iloc[checkOrigin + 1].to_numpy(dtype=float)
              - modelDf[dnsFactors].iloc[checkOrigin].to_numpy(dtype=float))
assert np.allclose(changeTargets[1].iloc[checkOrigin].to_numpy(dtype=float), handChange)
assert not predictorDf.isna().any().any()
print("predictors:", len(featureNames), " training rows available:", forecastOrigins[0] - maxHorizon)

predictors:

 33  training rows available: 4631


#### Training-only two stage tuning

Forty pseudo-origins a week apart, ending a full 20 days before the first evaluated origin, so every tuning outcome is observable before evaluation starts. Stage 1 picks (window, alpha) by pure VARX accuracy - a joint search would let weight-0 ties pick arbitrary VARX settings. Stage 2 picks the initial ensemble weight given those settings.

In [7]:
lastTuningOrigin = holdoutStart - maxHorizon
tuningOrigins = []
for step in range(numTuningOrigins - 1, -1, -1):
    tuningOrigins.append(lastTuningOrigin - tuningSpacing * step)
assert tuningOrigins[-1] + maxHorizon <= holdoutStart
assert tuningOrigins[0] > 250, "too little history before the first tuning origin"

selectedSettings = {}
tuningRows = []

for horizon in horizons:
    #quantities that do not depend on (window, alpha), computed once per tuning origin
    commonPerOrigin = []
    for origin in tuningOrigins:
        walkForecast = modelDf[observedCols].iloc[origin].to_numpy(dtype=float)
        actualYields = modelDf[observedCols].iloc[origin + horizon].to_numpy(dtype=float)
        residualForecast = forecastResiduals(origin, horizon)
        currentFactors = modelDf[dnsFactors].iloc[origin].to_numpy(dtype=float)
        commonPerOrigin.append([walkForecast, actualYields, residualForecast, currentFactors])

    tuningRmse = {}
    for windowLength in windowCandidates:
        for ridgeAlpha in ridgeCandidates:
            originResults = []
            for originIndex in range(len(tuningOrigins)):
                origin = tuningOrigins[originIndex]
                predictedChange = fitRidgeAndPredict(horizon, origin, windowLength, ridgeAlpha)
                walkForecast = commonPerOrigin[originIndex][0]
                actualYields = commonPerOrigin[originIndex][1]
                residualForecast = commonPerOrigin[originIndex][2]
                currentFactors = commonPerOrigin[originIndex][3]
                varxForecast = loadingMatrix @ (currentFactors + predictedChange) + residualForecast
                originResults.append([varxForecast, walkForecast, actualYields])

            for weight in weightCandidates:
                squaredErrors = []
                for result in originResults:
                    combined = weight * result[0] + (1.0 - weight) * result[1]
                    for error in (result[2] - combined):
                        squaredErrors.append(error ** 2)
                rmseBp = 100.0 * np.sqrt(np.mean(squaredErrors))
                tuningRmse[(windowLength, ridgeAlpha, weight)] = rmseBp
                if windowLength is None:
                    windowLabel = "expanding"
                else:
                    windowLabel = windowLength
                tuningRows.append([horizon, windowLabel, ridgeAlpha, weight, rmseBp])

    #stage 1: best (window, alpha) by pure VARX accuracy (weight 1.0)
    bestWindow = None
    bestAlpha = None
    bestVarxRmse = np.inf
    for windowLength in windowCandidates:
        for ridgeAlpha in ridgeCandidates:
            if tuningRmse[(windowLength, ridgeAlpha, 1.0)] < bestVarxRmse:
                bestVarxRmse = tuningRmse[(windowLength, ridgeAlpha, 1.0)]
                bestWindow = windowLength
                bestAlpha = ridgeAlpha

    #stage 2: best initial weight given those settings
    bestWeight = None
    bestWeightRmse = np.inf
    for weight in weightCandidates:
        if tuningRmse[(bestWindow, bestAlpha, weight)] < bestWeightRmse:
            bestWeightRmse = tuningRmse[(bestWindow, bestAlpha, weight)]
            bestWeight = weight

    selectedSettings[horizon] = {"window": bestWindow, "alpha": bestAlpha,
        "initialWeight": bestWeight, "varxTuningRmseBp": bestVarxRmse}
    print("horizon", horizon, "-> window:", bestWindow, " alpha:", bestAlpha,
          " initial weight:", bestWeight, " varx tuning rmse:", round(bestVarxRmse, 3), "bp")

tuningResults = pd.DataFrame(tuningRows,
    columns=["horizon", "window", "ridgeAlpha", "varxWeight", "tuningRmseBp"])

horizon 1 -> window: 504  alpha: 1000.0  initial weight: 0.0  varx tuning rmse: 2.911 bp


horizon 5 -> window: None  alpha: 100.0  initial weight: 0.5  varx tuning rmse: 5.038 bp


horizon 20 -> window: None  alpha: 1000.0  initial weight: 0.75  varx tuning rmse: 10.843 bp


#### Full holdout evaluation with the adaptive trailing weight

Three models at every origin: observed yield random walk, econ VARX, and the econ adaptive ensemble whose weight is re-chosen at every origin from the trailing realized record. Anti-leakage rule for the weight: an origin o can inform it only when o + h <= t.

In [8]:
realizedRecord = {}
for horizon in horizons:
    realizedRecord[horizon] = {}


def chooseTrailingWeight(horizon, currentOrigin, fallbackWeight):
    #only origins whose outcomes are realized may inform the weight
    usable = []
    for position in realizedRecord[horizon]:
        if position + horizon <= currentOrigin:
            usable.append(position)
    if len(usable) < minTrailing:
        return fallbackWeight
    usable = sorted(usable)
    recent = usable[-trailingWindow:]

    varxStack = []
    walkStack = []
    actualStack = []
    for position in recent:
        varxStack.append(realizedRecord[horizon][position][0])
        walkStack.append(realizedRecord[horizon][position][1])
        actualStack.append(realizedRecord[horizon][position][2])
    varxStack = np.array(varxStack)
    walkStack = np.array(walkStack)
    actualStack = np.array(actualStack)

    bestWeight = fallbackWeight
    bestLoss = np.inf
    for weight in weightCandidates:
        combined = weight * varxStack + (1.0 - weight) * walkStack
        loss = np.mean((actualStack - combined) ** 2)
        if loss < bestLoss:
            bestLoss = loss
            bestWeight = weight
    return bestWeight


forecastRows = []
originNumber = 0
for origin in forecastOrigins:
    originNumber = originNumber + 1
    for horizon in horizons:
        settings = selectedSettings[horizon]
        predictedChange = fitRidgeAndPredict(horizon, origin, settings["window"], settings["alpha"])
        currentFactors = modelDf[dnsFactors].iloc[origin].to_numpy(dtype=float)

        varxForecast = loadingMatrix @ (currentFactors + predictedChange) + forecastResiduals(origin, horizon)
        walkForecast = modelDf[observedCols].iloc[origin].to_numpy(dtype=float)

        ensembleWeight = chooseTrailingWeight(horizon, origin, settings["initialWeight"])
        ensembleForecast = ensembleWeight * varxForecast + (1.0 - ensembleWeight) * walkForecast

        targetPosition = origin + horizon
        actualYields = modelDf[observedCols].iloc[targetPosition].to_numpy(dtype=float)

        #recorded after the weight is chosen, so today's outcome can never inform today's weight
        realizedRecord[horizon][origin] = [varxForecast, walkForecast, actualYields]

        modelForecasts = [["Observed Yield Random Walk", walkForecast],
                          ["Econ VARX", varxForecast],
                          ["Econ Adaptive Ensemble", ensembleForecast]]
        for pair in modelForecasts:
            modelName = pair[0]
            predictedYields = pair[1]
            for matIndex in range(len(matList)):
                forecastRows.append({
                    "origin_number": originNumber,
                    "origin_position": origin,
                    "origin_date": modelDf.index[origin],
                    "target_date": modelDf.index[targetPosition],
                    "horizon": horizon,
                    "model": modelName,
                    "maturity": matList[matIndex],
                    "ensemble_weight": ensembleWeight,
                    "origin_yield": walkForecast[matIndex],
                    "actual_yield": actualYields[matIndex],
                    "predicted_yield": predictedYields[matIndex],
                })

    if originNumber % 200 == 0 or originNumber == len(forecastOrigins):
        print("finished origin", originNumber, "of", len(forecastOrigins))

yieldForecasts = pd.DataFrame(forecastRows)
assert len(yieldForecasts) == len(forecastOrigins) * 3 * 3 * 11
assert not yieldForecasts.isna().any().any()

finished origin 200 of 1149


finished origin 400 of 1149


finished origin 600 of 1149


finished origin 800 of 1149


finished origin 1000 of 1149


finished origin 1149 of 1149


#### Accuracy metrics and the head to head against final2

Same corrected metric definitions as final2: rmse/mae in bp, directional accuracy that skips unscorable rows (zero predicted change, or a realized change that is exactly zero from the 0.01 quantization) with the base rate shown, and DM tests with the rectangular kernel, HLN correction, and BH q-values. The decisive comparison is econ ensemble vs final2's ensemble on identical dates.

In [9]:
yieldForecasts["error_bp"] = 100.0 * (yieldForecasts["actual_yield"] - yieldForecasts["predicted_yield"])
yieldForecasts["actual_change"] = yieldForecasts["actual_yield"] - yieldForecasts["origin_yield"]
yieldForecasts["predicted_change"] = yieldForecasts["predicted_yield"] - yieldForecasts["origin_yield"]

modelNames = ["Observed Yield Random Walk", "Econ VARX", "Econ Adaptive Ensemble"]

#metrics with explicit loops, one row per model and horizon
metricRows = []
for modelName in modelNames:
    for horizon in horizons:
        rows = yieldForecasts[(yieldForecasts["model"] == modelName) & (yieldForecasts["horizon"] == horizon)]
        errors = rows["error_bp"].to_numpy()
        rmse = np.sqrt(np.mean(errors ** 2))
        mae = np.mean(np.abs(errors))

        predictedSign = np.sign(rows["predicted_change"].to_numpy())
        actualSign = np.sign(rows["actual_change"].to_numpy())
        #scorable rows need BOTH signs nonzero: the random walk predicts exactly zero change,
        #and quantized yields make some realized changes exactly zero (unwinnable ties)
        scorable = (predictedSign != 0) & (actualSign != 0)
        if scorable.sum() > 0:
            directionalAccuracy = np.mean(predictedSign[scorable] == actualSign[scorable])
        else:
            directionalAccuracy = np.nan

        nonzeroActual = actualSign[actualSign != 0]
        upShare = np.mean(nonzeroActual > 0)
        baseRate = max(upShare, 1.0 - upShare)

        metricRows.append([modelName, horizon, rmse, mae, directionalAccuracy, baseRate, len(rows)])

curveMetrics = pd.DataFrame(metricRows,
    columns=["model", "horizon", "curveRmseBp", "curveMaeBp", "directionalAccuracy", "daBaseRate", "forecasts"])
curveMetrics = curveMetrics.sort_values(["horizon", "curveRmseBp"])
print("shared holdout:", len(forecastOrigins), "origins")
display(curveMetrics.round(4))


def dieboldMariano(lossA, lossB, horizon):
    #classic DM with rectangular kernel through lag h-1 and the HLN small sample correction
    difference = np.asarray(lossA) - np.asarray(lossB)
    n = len(difference)
    centered = difference - difference.mean()
    maxLag = min(horizon - 1, n - 2)
    variance = np.dot(centered, centered) / n
    for lag in range(1, maxLag + 1):
        variance = variance + 2.0 * np.dot(centered[lag:], centered[:-lag]) / n
    if variance <= 0.0:
        return np.nan, np.nan
    statistic = difference.mean() / np.sqrt(variance / n)
    correction = (n + 1 - 2 * horizon + horizon * (horizon - 1) / n) / n
    statistic = statistic * np.sqrt(correction)
    pValue = 2.0 * stats.t.sf(abs(statistic), df=n - 1)
    return statistic, pValue


#align final3 and final2 ensembles on identical (origin_date, horizon, maturity)
final2Ensemble = final2Forecasts[final2Forecasts["model"] == "Adaptive Ensemble"]
final2Ensemble = final2Ensemble[["origin_date", "horizon", "maturity", "predicted_yield", "actual_yield"]]
final2Ensemble = final2Ensemble.rename(columns={"predicted_yield": "final2Predicted"})

final3Ensemble = yieldForecasts[yieldForecasts["model"] == "Econ Adaptive Ensemble"]
final3Ensemble = final3Ensemble[["origin_date", "horizon", "maturity", "predicted_yield"]]
final3Ensemble = final3Ensemble.rename(columns={"predicted_yield": "final3Predicted"})

headToHead = final3Ensemble.merge(final2Ensemble, on=["origin_date", "horizon", "maturity"])
assert len(headToHead) == len(final3Ensemble), "origin alignment with final2 failed"
headToHead["final3SqBp"] = (100.0 * (headToHead["actual_yield"] - headToHead["final3Predicted"])) ** 2
headToHead["final2SqBp"] = (100.0 * (headToHead["actual_yield"] - headToHead["final2Predicted"])) ** 2

dmRows = []
for horizon in horizons:
    h2h = headToHead[headToHead["horizon"] == horizon]
    perOrigin = h2h.groupby("origin_date")[["final3SqBp", "final2SqBp"]].mean().sort_index()
    statistic, pValue = dieboldMariano(perOrigin["final3SqBp"], perOrigin["final2SqBp"], horizon)
    dmRows.append([horizon, "Econ Ensemble (final3)", "Adaptive Ensemble (final2)", statistic, pValue])

    f3Rows = yieldForecasts[(yieldForecasts["model"] == "Econ Adaptive Ensemble") & (yieldForecasts["horizon"] == horizon)]
    rwRows = yieldForecasts[(yieldForecasts["model"] == "Observed Yield Random Walk") & (yieldForecasts["horizon"] == horizon)]
    f3Loss = (f3Rows["error_bp"] ** 2).groupby(f3Rows["origin_date"]).mean().sort_index()
    rwLoss = (rwRows["error_bp"] ** 2).groupby(rwRows["origin_date"]).mean().sort_index()
    statistic, pValue = dieboldMariano(f3Loss, rwLoss, horizon)
    dmRows.append([horizon, "Econ Ensemble (final3)", "Observed Yield Random Walk", statistic, pValue])

dmResults = pd.DataFrame(dmRows, columns=["horizon", "modelA", "modelB", "dmStatistic", "pValue"])

#BH q-values across the six tests, cite these when claiming significance
finiteMask = dmResults["pValue"].notna().to_numpy()
qValues = np.full(len(dmResults), np.nan)
qValues[finiteMask] = stats.false_discovery_control(dmResults["pValue"].to_numpy()[finiteMask])
dmResults["qValueBH"] = qValues
display(dmResults.round(4))

shared holdout: 1149 origins


,model,horizon,curveRmseBp,curveMaeBp,directionalAccuracy,daBaseRate,forecasts
0,Observed Yield Random Walk,1,6.1118,4.1453,NaN,0.5138,12639
6,Econ Adaptive Ensemble,1,6.1218,4.1667,0.5284,0.5138,12639
3,Econ VARX,1,6.1437,4.2027,0.5269,0.5138,12639
7,Econ Adaptive Ensemble,5,13.1543,9.1809,0.5629,0.5575,12639
1,Observed Yield Random Walk,5,13.1950,9.1995,NaN,0.5575,12639
4,Econ VARX,5,13.4115,9.4512,0.5375,0.5575,12639
8,Econ Adaptive Ensemble,20,27.0082,19.8932,0.5385,0.5704,12639
5,Econ VARX,20,27.6808,20.6469,0.5293,0.5704,12639
2,Observed Yield Random Walk,20,28.0288,20.4367,NaN,0.5704,12639


,horizon,modelA,modelB,dmStatistic,pValue,qValueBH
0,1,Econ Ensemble (final3),Adaptive Ensemble (final2),2.0203,0.0436,0.2615
1,1,Econ Ensemble (final3),Observed Yield Random Walk,0.6296,0.5291,0.6349
2,5,Econ Ensemble (final3),Adaptive Ensemble (final2),1.3424,0.1797,0.4024
3,5,Econ Ensemble (final3),Observed Yield Random Walk,-0.4240,0.6716,0.6716
4,20,Econ Ensemble (final3),Adaptive Ensemble (final2),0.6348,0.5257,0.6349
5,20,Econ Ensemble (final3),Observed Yield Random Walk,-1.2788,0.2012,0.4024


#### Save results

Positive dmStatistic in the final3-vs-final2 rows means final2 has the lower loss. The verdict from the tables above: final2 wins at every horizon, so final2 remains the paper's headline model and this notebook documents that the economics features are spanned.

In [10]:
resultsDir = dataDir / "generated" / "final3_results"
resultsDir.mkdir(parents=True, exist_ok=True)
yieldForecasts.to_csv(resultsDir / "yield_forecasts.csv", index=False)
curveMetrics.to_csv(resultsDir / "curve_metrics.csv", index=False)
tuningResults.to_csv(resultsDir / "training_only_tuning.csv", index=False)
dmResults.to_csv(resultsDir / "dm_vs_final2_and_rw.csv", index=False)
print("saved final3 results to", resultsDir)

saved final3 results to ..\data\generated\final3_results
